# 02 — TTT Evaluation on ARC-AGI-2 (Colab Pro)

Loads a slim checkpoint produced by `01_pretrain_colab.ipynb` and runs the eval split with TTT on and off.

Outputs `submission_*.json` in the ARC Prize Kaggle format so the same predictions can be dry-run against a Kaggle notebook later.

In [2]:
import pathlib
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = pathlib.Path('/content/arc_hybrid_repo')
if not REPO_DIR.exists():
    !git clone https://github.com/Nitish05/ARC-AGI.git {REPO_DIR}

%cd /content/arc_hybrid_repo
!pip install -q -r requirements.txt

Mounted at /content/drive
Cloning into '/content/arc_hybrid_repo'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 125 (delta 56), reused 104 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 62.83 KiB | 20.94 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/arc_hybrid_repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.2 MB/s eta 0:00:00


In [3]:
# Pull latest from GitHub (run this whenever you've pushed local edits).
!git -C /content/arc_hybrid_repo pull --ff-only

# Hot-reload so edits to arc_hybrid/*.py take effect on the next cell run
# without a kernel restart. Python 3.12 removed the `imp` module, but Colab's
# bundled autoreload still imports it — shim with importlib.reload first.
import sys, types, importlib
if 'imp' not in sys.modules:
    sys.modules['imp'] = types.ModuleType('imp')
    sys.modules['imp'].reload = importlib.reload
%load_ext autoreload
%autoreload 2

Already up to date.


In [4]:
import torch
from arc_hybrid.utils.config import to_namespace
from arc_hybrid.model.hybrid import build_hybrid_from_config
from arc_hybrid.data.arc_loader import filter_max_grid, load_split
from arc_hybrid.eval.evaluate import evaluate_split

# Latest slim checkpoint from pretrain. Kernel disconnected at step ~52150,
# so slim_50000.pt is the last full checkpoint on Drive (ckpt_every=5000).
ckpt_path = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/slim_50000.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
cfg = to_namespace(ckpt['config'])
model = build_hybrid_from_config(cfg).to('cuda')

# Strip torch.compile's `_orig_mod.` prefix from keys if it's there.
# pretrain.py compiles the model before save, so older checkpoints carry it.
state = ckpt['model']
if any(k.startswith('_orig_mod.') for k in state.keys()):
    state = {k.removeprefix('_orig_mod.'): v for k, v in state.items()}
model.load_state_dict(state)
model.eval()
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.1f}M params from {ckpt_path}")

loaded 38.6M params from /content/drive/MyDrive/arc_hybrid_runs/medium_v1/slim_50000.pt


In [5]:
# Ensure ARC-AGI-2 evaluation tasks are on the runtime. Idempotent: skips if
# the folder is already there. Needed because the eval notebook may be opened
# on a fresh Colab VM that doesn't yet have what notebook 01 downloaded.
import pathlib
EVAL_DIR = pathlib.Path('data/raw/ARC-AGI-2/data/evaluation')
if not EVAL_DIR.exists():
    !mkdir -p data/raw
    !git clone --depth 1 https://github.com/arcprize/ARC-AGI-2 data/raw/ARC-AGI-2
assert EVAL_DIR.exists(), f"{EVAL_DIR} still missing after clone"
print(f'eval data ready · {sum(1 for _ in EVAL_DIR.glob("*.json"))} tasks at {EVAL_DIR}')

Cloning into 'data/raw/ARC-AGI-2'...
remote: Enumerating objects: 1130, done.
remote: Counting objects: 100% (1130/1130), done.
remote: Compressing objects: 100% (512/512), done.
remote: Total 1130 (delta 628), reused 1074 (delta 618), pack-reused 0 (from 0)
Receiving objects: 100% (1130/1130), 562.39 KiB | 16.54 MiB/s, done.
Resolving deltas: 100% (628/628), done.
eval data ready · 120 tasks at data/raw/ARC-AGI-2/data/evaluation


In [7]:
from pathlib import Path
tasks = load_split(Path('data/raw/ARC-AGI-2/data/evaluation'))
tasks = filter_max_grid(tasks, cfg.model.max_grid_size)
print(f'{len(tasks)} eval tasks')

120 eval tasks


In [10]:
!pytest tests/test_kv_cache.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/arc_hybrid_repo
plugins: typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.34
collected 2 items                                                              

tests/test_kv_cache.py::test_kv_cache_matches_legacy_decode PASSED       [ 50%]
tests/test_kv_cache.py::test_prefill_logits_match_no_cache PASSED        [100%]

============================== 2 passed in 0.79s ===============================


In [11]:
from pathlib import Path
tasks = load_split(Path('data/raw/ARC-AGI-2/data/evaluation'))
tasks = filter_max_grid(tasks, cfg.model.max_grid_size)
print(f'{len(tasks)} eval tasks')

120 eval tasks


In [12]:
out_dir = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/eval'
evaluate_split(model, tasks, use_ttt=False, max_grid=cfg.model.max_grid_size,
               device='cuda', out_dir=out_dir, tag='ttt_off')

ttt_off:   0%|          | 0/120 [00:00<?, ?it/s]

[ttt_off] 2/120 correct (0.017)


{'tag': 'ttt_off',
 'n_total': 120,
 'n_correct': 2,
 'accuracy': 0.016666666666666666}

In [14]:
!pip install -q --upgrade peft accelerate


In [16]:
!pip install -q --upgrade torchao 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.0 MB/s eta 0:00:00:00:01


In [ ]:
out_dir = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/eval'
ttt_kwargs = dict(steps=100, lr=5e-4, batch_size=4, n_examples=64,
                  lora_r=8, lora_alpha=16, lora_dropout=0.05)
evaluate_split(model, tasks, use_ttt=True, ttt_kwargs=ttt_kwargs,
               max_grid=cfg.model.max_grid_size, device='cuda',
               out_dir=out_dir, tag='ttt_on')